# 《跨境底牌：硅深暗战》/ *Global Edge* — Colab 96GB G4 双语多变体商战广播剧工作台 (`Fish Audio Drama 3`)
### Zero-API Open-Weight Golden Stack (`v2_real_human_studio` · 0% Commercial Voice API · 0% Synthetic TTS)

1. **8 角色真人棚录近讲声纹底座 (`v2_real_human_studio` · 44.1kHz Mono `-16 LUFS`)**：
   - **Core Cast (100% E001 / 96.0% Season 1 EN)**: `lin_chen` (同声源硅谷美音+冷峻中文双语主角), `zhou_mo` (深圳 CTO 普通话+硬核英文术语), `victoria_vance` (硅谷沙丘路 VC 美音), `alistair_finch` (伦敦 RP 零售副总裁英音), `claire_chen_gc` (纽约总法务美音)。
   - **Controlled International Guests (4.0% Season 1 EN · E011/E013/E016)**: `rajiv_menon` (印度精英高管), `wei_ling_tan` (新加坡主权基金合伙人), `henrik_lindqvist` (北欧欧盟法务)。
   - 生产模式强制执行 `validate_production_voice_cast()`，严禁任何合成/占位声纹进入 GPU 推理。自定义真人干声可直接投放到云盘 `custom_anchors/<role_id>.{wav,mp3,m4a,flac}`。
2. **第一期四变体同台盲测 (`AUDITION,E001_VARIANTS` = `AUDITION_GLOBAL_EDGE` + `E001_VA` / `E001_VB` / `E001_VC` / `E001_VD`)**：
   - 同一 5 条语料种子、同一 8 条高管短语、同一核心商战危机，一键对比四种叙事/教学节奏与声场编排。
3. **96GB G4 三模型同卡常驻 + `0.80x` 确定性缓速复用**：
   - `fishaudio/drama-3` (35B MoE FP8 + RAS) + `MOSS-SoundEffect-v2.0` (48kHz 商战纯拟音) + `ACE-Step 1.5 XL` (HBO 冷峻室内乐纯器乐) + CPU `faster-whisper-small-int8` (0 MB 显存双脚本回听质检)。
   - Act 5 片尾 3 分钟黑话拆局的 `0.80x` 慢速拆解自动复用 `1.0x` Drama 3 缓存干声，由 `mixer.py` 经 FFmpeg 不降调 `atempo=0.800` 与 `-16 LUFS` 三母线混音并导出中英双行零重叠 `.srt`。


## 🚀 Mode A（推荐 / Recommended）：Antigravity Agent HTTPS 桥接全自动托管
> **使用方法 / How to use**：
> 1. 点击下方 `▶` 挂载 Google Drive 并启动加密 HTTPS 桥接（`colab_agent_bridge.py`）。
> 2. 看到终端输出就绪提示后，回到对话框告知 Agent，由 Agent 自动完成声纹验收、渲染 `AUDITION,E001_VARIANTS` 并核验每一条成片。
> 3. 全部母带原子同步回 Google Drive 后，如检测到 `/content/.colab_auto_unassign` 标记，Notebook 会自动调用 `runtime.unassign()` 释放 G4 实例立停计费。


In [ ]:
# @title ▶ Mode A: Start Antigravity Agent HTTPS Bridge (G4 96GB)
import os, shutil, subprocess, urllib.request
from pathlib import Path

if os.path.isdir('/content/drive/My Drive') and not os.path.exists('/content/drive/MyDrive'):
    os.symlink('/content/drive/My Drive', '/content/drive/MyDrive')
if not Path('/content/drive/MyDrive').exists():
    from google.colab import drive
    drive.mount('/content/drive')

DRIVE_STUDIO_DIR = Path(
    '/content/drive/MyDrive/《跨境底牌：硅深暗战》Global_Edge_双语商战广播剧/01_Colab_96GB_FishDrama3_工作台'
)
DRIVE_STUDIO_DIR.mkdir(parents=True, exist_ok=True)
(DRIVE_STUDIO_DIR / 'custom_anchors').mkdir(parents=True, exist_ok=True)

bridge_script = Path('/content/colab_agent_bridge.py')
if (DRIVE_STUDIO_DIR / 'colab_agent_bridge.py').is_file():
    shutil.copy2(DRIVE_STUDIO_DIR / 'colab_agent_bridge.py', bridge_script)
else:
    url = 'https://raw.githubusercontent.com/riversxiao/aigc-audio-pipeline/main/colab_96gb/colab_agent_bridge.py'
    urllib.request.urlretrieve(url, str(bridge_script))

bridge_json = DRIVE_STUDIO_DIR / 'colab_agent_bridge.json'
subprocess.run(
    ['python3', '-u', str(bridge_script), '--bridge-file', str(bridge_json)],
    check=False,
)

if Path('/content/.colab_auto_unassign').exists():
    print('🛑 全部成片已同步至 Google Drive，正在主动销毁 Colab G4 运行时立停计费...')
    from google.colab import runtime
    runtime.unassign()


## ⚡ Mode B（自助一键直跑 / Direct One-Click Batch Run）：渲染 `AUDITION,E001_VARIANTS` 并原子同步云盘
> 可按需修改下方参数（如仅跑 `E001_VA,E001_VB` 或全季 `E001-E024`）。默认 `EPISODES = "AUDITION,E001_VARIANTS"`，`PROFILE = "quality"`，`AUTO_UNASSIGN = True`。


In [ ]:
# @title ▶ Mode B: Direct One-Click Render (AUDITION + E001_VA..E001_VD)
import os, subprocess
from pathlib import Path

EPISODES = 'AUDITION,E001_VARIANTS'  # @param {type:'string'}
PROFILE = 'quality'  # @param ['quality', 'fp8', 'fast']
SKIP_EXISTING = True  # @param {type:'boolean'}
MIRROR_TO_MAIN_MASTERS = True  # @param {type:'boolean'}
AUTO_UNASSIGN = True  # @param {type:'boolean'}

if os.path.isdir('/content/drive/My Drive') and not os.path.exists('/content/drive/MyDrive'):
    os.symlink('/content/drive/My Drive', '/content/drive/MyDrive')
if not Path('/content/drive/MyDrive').exists():
    from google.colab import drive
    drive.mount('/content/drive')

REPO = Path('/content/aigc-audio-pipeline')
if not REPO.exists():
    subprocess.check_call(['git', 'clone', 'https://github.com/riversxiao/aigc-audio-pipeline.git', str(REPO)])
else:
    subprocess.check_call(['git', '-C', str(REPO), 'pull', '--ff-only'])

cmd = [
    'python3', '-u', str(REPO / 'colab_96gb/launch_g4_global_edge_drama3.py'),
    '--episodes', EPISODES,
    '--profile', PROFILE,
]
if SKIP_EXISTING:
    cmd.append('--skip-existing')
if MIRROR_TO_MAIN_MASTERS:
    cmd.append('--mirror-to-main-masters')
if AUTO_UNASSIGN:
    cmd.append('--auto-unassign')

print('🚀 Launching:', ' '.join(cmd))
subprocess.check_call(cmd, cwd=str(REPO))

if Path('/content/.colab_auto_unassign').exists():
    print('🛑 全部成片已同步至 Google Drive，正在主动销毁 Colab G4 运行时立停计费...')
    from google.colab import runtime
    runtime.unassign()


## 🎧 Post-Run Audition & A/B Variant Player / 渲染完成后云盘验收与在线盲听
> 检查 Google Drive 同步状态，并在 Notebook 内直接试听 `AUDITION_GLOBAL_EDGE` 与 `E001_VA` / `E001_VB` / `E001_VC` / `E001_VD` 成片 MP3。


In [ ]:
# @title ▶ Check Google Drive Masters & Inline Audio Player (AUDITION + E001_VA..VD)
from pathlib import Path
from IPython.display import Audio, display

DRIVE_SHOW = Path('/content/drive/MyDrive/《跨境底牌：硅深暗战》Global_Edge_双语商战广播剧')
SEARCH_DIRS = [
    DRIVE_SHOW / '02_分集双语成片母带_WAV_MP3_SRT',
    DRIVE_SHOW / '01_Colab_96GB_FishDrama3_工作台' / 'outputs',
]

seen = set()
mp3_files = []
for d in SEARCH_DIRS:
    if d.is_dir():
        for f in sorted(d.glob('*.mp3')):
            if f.name not in seen:
                seen.add(f.name)
                mp3_files.append(f)

if not mp3_files:
    print('ℹ️ 暂未在 Google Drive 检测到成片 MP3，请先运行 Mode A 或 Mode B。')
else:
    print(f'✅ 共发现 {len(mp3_files)} 个已归档双语母带 MP3：')
    for mp3 in mp3_files:
        srt = mp3.with_suffix('.srt')
        sz_mb = mp3.stat().st_size / (1024 * 1024)
        print(f'\n🎧 {mp3.name} ({sz_mb:.2f} MB) | SRT: {"OK" if srt.is_file() else "N/A"}')
        display(Audio(filename=str(mp3)))


## 🛑 Manual Runtime Release Helper / 手动释放 G4 运行时立停计费
> 听验完毕确认所有文件已安全落盘 Google Drive 后，点击下方按钮立即释放算力。


In [ ]:
# @title ▶ Release Colab G4 Runtime Immediately
import os
from google.colab import runtime

os.sync()
print('🛑 Releasing Colab G4 runtime now...')
runtime.unassign()
